# Mind Quest: from one player to a group

Mind Quest is an educational jump-and-run game about attention, response inhibition and executive control. While playing, a player completes three classic experiments: a **flanker task** (Eriksen & Eriksen, 1974; Rueda et al., 2004), a **stop-signal task** (Logan & Cowan, 1984; Verbruggen et al., 2019) and a **2-back task** (Kirchner, 1958), plus a short questionnaire about screen habits, gaming and alertness.

**Research question.** Is more daily screen time linked to weaker attentional control, measured as a larger flanker effect? And, since one game session gives data for only one person: **how many players would a study with this game need to detect such a link?**

One playthrough tells us about one person. This notebook turns it into data a statistical analysis can work with:

1. **Play** the game: the experiments save the player's data in `data/`.
2. **Bootstrap** the player's trials: how precise are their scores?
3. **Simulate** a group of players whose typical scores come from the bootstrap and whose differences come from the literature.
4. **Analyse** the simulated group: does the analysis recover the effects built into the simulation, and how much power would a real study have?

All code lives in `src/`; this notebook only calls its high-level functions. Run it from the project folder, from top to bottom (about 5 to 10 minutes without the game).

## Setup

Install the requirements once with `pip install -r requirements.txt` (see `README.md`).

In [ ]:
from src.analysis import (
    aggregate_full_study_data,
    analyze_and_plot_flanker,
    analyze_and_plot_nback,
    analyze_and_plot_stopsignal,
    analyze_and_plot_tower,
    analyze_screen_modalities,
    plot_individual_media_impact,
    run_full_questionnaire_regression,
    run_power_analysis,
)
from src.bootstrap import bootstrap_session, save_bootstrap
from src.data import clear_game_progress, load_session
from src.game import run_game
from src.simulation import SIMULATED_DIR, Effects, simulate_participants

## 1. Play the game

Running the cell below opens the game window. It starts with the questionnaire; after that the levels open one after the other. The experiments are in Level 1 (flanker), Level 2 (stop-signal) and Level 3 (2-back and Tower of London). Close the window to come back to the notebook; every finished experiment has been saved in `data/`.

* `PARTICIPANT_ID`: an anonymous label for the data files. `None` creates a random label such as `p-3fa2c1`.
* `NEW_PARTICIPANT = True` first clears the unlocked levels and ghost races of the previous player (the data in `data/` are kept).
* `PLAY_GAME = False` skips the game and uses the data of whoever played an experiment last. If nobody has (for example in a fresh clone), the demo playthrough in `data/example/` is used. It was recorded by an automated test player with scripted responses, not by a person, so it shows how the analysis works rather than how real players perform.

Admin mode is switched off (`allow_admin=False`), so no experiment can be skipped.

In [ ]:
PARTICIPANT_ID = None
NEW_PARTICIPANT = True
PLAY_GAME = True

if PLAY_GAME:
    if NEW_PARTICIPANT:
        clear_game_progress()
    run_game(participant_id=PARTICIPANT_ID, allow_admin=False)

## 2. Bootstrap the player's data

Each score rests on a few dozen trials, so it would come out somewhat differently if the player played again. The bootstrap (Efron, 1979) estimates how much: the player's trials are drawn again **with replacement** (within each condition, so the design of the task stays intact), the score is recomputed, and this is repeated 5,000 times. The spread of these values is the standard error of the score, and their middle 95% is a confidence interval.

We use the most recent player who completed at least one of the flanker, stop-signal and 2-back tasks, their most recent session and their **first** attempt at every experiment (later attempts are affected by practice); practice trials are left out. The bootstrapped measures are:

| Task | Measures |
| --- | --- |
| Flanker | baseline RT (congruent trials), flanker effect (incongruent minus congruent RT; Eriksen & Eriksen, 1974), trial-to-trial noise, error rates |
| Stop-signal | mean go RT, its noise, choice errors, and the stop-signal reaction time (SSRT, integration method; Verbruggen et al., 2019) |
| 2-back | sensitivity d′ (log-linear correction; Hautus, 1995), response criterion *c* (Stanislaw & Todorov, 1999), hit RTs |

The SSRT is the least precise measure: the game has 16 stop trials, while Verbruggen et al. (2019) recommend at least 50. Tasks the player did not complete are skipped, and the simulation then uses published values instead. The standard errors are used in the next step: they decide how much the player's scores shape the simulated group.

The results are saved in `data/simulated/`: `bootstrap_summary.csv` (estimate, standard error and 95% CI per measure) and `bootstrap_samples.csv` (all 5,000 bootstrap values per measure).

In [ ]:
session = load_session(PARTICIPANT_ID)
participant = session["participant_id"]
completed = [task for task in session if task not in ("participant_id", "source")]
print(f"Player: {participant} (from {session['source']}); completed: {', '.join(completed) or 'nothing'}")

boot = bootstrap_session(session, n_resamples=5000, seed=2026)
for path in save_bootstrap(boot, participant) or ["Nothing to bootstrap: the simulation will use published values only."]:
    print(path)

## 3. Simulate a group of players

Every simulated player answers the questionnaire and completes every trial of the flanker, stop-signal and 2-back tasks, in exactly the format the game writes, so the same analysis code works on real and simulated data. Each player first gets their own true scores, drawn from normal distributions:

| Score | Mean | How much people differ (SD) |
| --- | --- | --- |
| Flanker baseline RT and flanker effect | player and Kaufman et al. (2016), weighted by precision | Kaufman et al. (2016), Table 2, young adults |
| Go RT and SSRT | player and Congdon et al. (2014), weighted by precision | Congdon et al. (2014), Table 1, healthy controls |
| 2-back d′ | player and Forsberg et al. (2020), weighted by precision | Forsberg et al. (2020), Table 3, younger adults |
| Daily screen time | Sedaghat et al. (2025) | Sedaghat et al. (2025) |

**How the player and the literature are combined.** The player's score is one noisy measurement; the published mean describes the population. Following the normal model with known variances (Gelman et al., 2013, Section 2.5), both are averaged with weights proportional to their precision (1 / variance). The player's weight is SD² / (SD² + SE²), with SD the between-person SD from the literature and SE the bootstrap standard error. A precisely measured player (small SE) moves the group towards their own score; a noisy one (large SE) leaves it close to the published mean. The weights are saved with the other values in `parameters_simulated.csv` (`player_weight`). For tasks the player did not complete, the published means are used. Trial-to-trial noise and error rates have no published between-person SD and take the player's own estimate.

**The group is based on one player's session.** The simulated players are centred on one person's scores (as far as their precision allows), not on a sample of real people. They show what a group *could* look like, which is useful for planning and checking an analysis, but they are no substitute for collecting data from many players.

Values without a published source (error rates, the 2-back criterion and RTs, and the answer distributions of the 1–5 questions) are marked as assumptions in `src/simulation.py`. Screen time comes from a study that asked about all screen use, including study and work, while the game asks about free time only, so the simulated screen times are probably too high.

The trials then follow a model of each task: RT = baseline + flanker effect + noise in the flanker task, a race between going and stopping in the stop-signal task (Logan & Cowan, 1984), and signal detection theory in the 2-back task (Stanislaw & Todorov, 1999).

**The effects below are assumptions, not findings.** By default every hour of screen time above the mean adds 5 ms to the flanker effect and to the SSRT, and higher alertness speeds up responses. A simulation only contains the effects that are put into it: it is a tool to plan and check an analysis, not evidence that the effects exist. Set an effect to 0 to simulate a world without it.

Everything is saved in `data/simulated/`: one file per task (`questionnaire_simulated.csv`, `flanker_simulated.csv`, `stopsignal_simulated.csv`, `nback_simulated.csv`), the true scores of every simulated player (`true_values_simulated.csv`) and the population values used, with their sources (`parameters_simulated.csv`).

In [ ]:
effects = Effects(
    screen_time_on_flanker_ms=5.0,
    screen_time_on_ssrt_ms=5.0,
    alertness_on_rt_ms=-15.0,
)
sim = simulate_participants(n=50, bootstrap=boot, participant_id=participant, seed=2026, effects=effects)
for path in sim.save(SIMULATED_DIR):
    print(path)

## 4. Statistical analysis

All analyses use the simulated group `sim` from section 3. A simulation contains exactly the effects that were put into it (`effects` above), so the analysis answers two questions:

* **Does the analysis work?** It should find the built-in screen-time effect and should *not* find effects that were not built in.
* **How many players would a real study need?** A power analysis repeats the simulated study many times for different sample sizes.

The results depend on who played in section 1 and on the settings above, so the text below explains how to read them; the numbers are in the printed tables and figures.

### 4.1 Individual debriefs

Each debrief places one player's scores against the published norms: the dashed curve is the population distribution and the shaded band the norm ± 1 SD (Kaufman et al., 2016; Congdon et al., 2014; Forsberg et al., 2020). We show the simulated player `sim-001`; the real player's scores and their bootstrap confidence intervals are in section 2. Only real players have Tower of London data (Shallice, 1982), so that debrief appears for the player from section 1 if they solved the puzzles.

In [ ]:
example_player = "sim-001"
fig_flanker = analyze_and_plot_flanker(example_player, sim)
fig_stop = analyze_and_plot_stopsignal(example_player, sim)
fig_nback = analyze_and_plot_nback(example_player, sim)

fig_tower = analyze_and_plot_tower(participant, data_dir=session["source"]) if participant else None
if fig_tower is None:
    print("No Tower of London data for the player from section 1.")

### 4.2 Does screen time predict the flanker effect?

A multiple regression (ordinary least squares) predicts each simulated player's flanker effect from all five questionnaire answers. Given the settings above, we expect:

* **screen time:** a slope close to the built-in effect (`effects.screen_time_on_flanker_ms` per hour);
* **all other answers:** no effect. Alertness speeds up all responses equally, which cancels out in the flanker effect (incongruent minus congruent), and short-video use, gaming and interest in psychology have no built-in effect at all.

In [ ]:
full_cohort = aggregate_full_study_data(sim)
full_regression = run_full_questionnaire_regression(full_cohort)
print(full_regression.summary().tables[1])

estimate = full_regression.params["screen_time_hours"]
low, high = full_regression.conf_int().loc["screen_time_hours"]
print(f"\nBuilt-in screen-time effect: {effects.screen_time_on_flanker_ms:+.1f} ms per hour; "
      f"estimated: {estimate:+.1f} ms per hour, 95% CI [{low:.1f}, {high:.1f}], "
      f"p = {full_regression.pvalues['screen_time_hours']:.3f}")

**How to read this.** If the 95% confidence interval of the screen-time slope contains the built-in value, the analysis recovers the effect. With 50 players the interval is wide, so the slope can be non-significant even though the effect is really there: that is a matter of power (section 4.5), not of a wrong analysis. For the other answers, a significant slope would be a false positive; at α = .05 about one in twenty such tests is significant by chance.

### 4.3 Comparing screen habits on one scale

The same model with standardised variables (z-scores) puts screen time, short-video use, gaming and alertness on one scale. Points show the standardised slopes (β*) with 95% confidence intervals; red points are significant (p < .05).

Only screen time has a built-in effect. Short-video use was simulated to correlate with screen time (ρ = 0.5 on the latent scale), so the two share variance and their intervals are wider. Differences between the habits in this figure reflect the simulation's assumptions, not findings about real media use.

In [ ]:
modality_model, fig_modality = analyze_screen_modalities(full_cohort)
print(modality_model.summary().tables[1])

### 4.4 Personal media footprint

The regression from 4.2 applied to one player: for each habit, the coefficient times the player's distance from the group mean gives that habit's model-implied contribution to their flanker effect. These contributions follow from the simulation's assumptions and from a model fitted to 50 simulated players; they are not causal effects of the player's habits.

In [ ]:
fig_personal = plot_individual_media_impact(example_player, sim, simulated_cohort=sim)

### 4.5 Power analysis: how many players would a study need?

The simulated study is repeated `N_REPLICATIONS` times for each sample size, with the same effects and the same real player as above. Each time, the regression from 4.2 is fitted; the power is the share of repetitions in which the screen-time slope is positive and significant (p < .05). With 100 repetitions each power estimate is precise to about ±5 percentage points; this takes a few minutes.

In [ ]:
N_REPLICATIONS = 100

power_results, fig_power = run_power_analysis(
    bootstrap_results=boot,
    sample_sizes=(25, 50, 100, 175, 250, 325),
    n_replications=N_REPLICATIONS,
    effects=effects,
)
print(power_results.to_string(index=False))
enough = power_results.loc[power_results["power"] >= 0.8, "sample_size"]
print(f"\nSmallest sample size with at least 80% power: {enough.min() if len(enough) else 'none of the sizes tried'}")

### 4.6 Interpretation and discussion

**Answer to the research question.** The power curve shows how many players a study with this game would need to detect the assumed screen-time effect with 80% power. The main reason so many players are needed is the short tasks: each player completes only 12 incongruent and 12 congruent flanker trials, so every individual flanker effect is noisy (section 2 shows how wide one player's confidence interval is). A real study could need fewer players if each played longer versions of the tasks. The simulated players also inherit the real player's trial-to-trial noise (section 3): if that player's reaction times were very variable, even the largest sample size may stay below 80% power, which means the game's tasks are too short to detect an effect of this size.

**Limitations.**

* The data are simulated: they show what a study *could* find under our assumptions, not what is true for real players.
* The simulated group is centred on one player's session, weighted against the literature (section 3).
* The size of the screen-time effect (5 ms per hour by default) is an assumption, not a published estimate; with a smaller true effect, even more players would be needed.
* The simulation simplifies the tasks: reaction times are normally distributed, errors are independent of reaction times, and the stop-signal task has only 16 stop trials, fewer than the 50 that Verbruggen et al. (2019) recommend.
* The published screen times (Sedaghat et al., 2025) include study and work, so the simulated leisure screen times are probably too high.

**Next step.** Collect real data with the game and run the same analyses on them: `src.data.load_task` loads the real players' files in the same format as the simulated ones.

## References

Congdon, E., Altshuler, L. L., Mumford, J. A., Karlsgodt, K. H., Sabb, F. W., Ventura, J., McGough, J. J., London, E. D., Cannon, T. D., Bilder, R. M., & Poldrack, R. A. (2014). Neural activation during response inhibition in adult attention-deficit/hyperactivity disorder: Preliminary findings on the effects of medication and symptom severity. *Psychiatry Research: Neuroimaging, 222*, 17–28. https://doi.org/10.1016/j.pscychresns.2014.02.002

Efron, B. (1979). Bootstrap methods: Another look at the jackknife. *The Annals of Statistics, 7*(1), 1–26. https://doi.org/10.1214/aos/1176344552

Eriksen, B. A., & Eriksen, C. W. (1974). Effects of noise letters upon the identification of a target letter in a nonsearch task. *Perception & Psychophysics, 16*(1), 143–149.

Forsberg, A., Fellman, D., Laine, M., Johnson, W., & Logie, R. H. (2020). Strategy mediation in working memory training in younger and older adults. *Quarterly Journal of Experimental Psychology, 73*(8), 1206–1226. https://doi.org/10.1177/1747021820915107

Gelman, A., Carlin, J. B., Stern, H. S., Dunson, D. B., Vehtari, A., & Rubin, D. B. (2013). *Bayesian data analysis* (3rd ed.). CRC Press.

Hautus, M. J. (1995). Corrections for extreme proportions and their biasing effects on estimated values of d'. *Behavior Research Methods, Instruments, & Computers, 27*(1), 46–51.

Kaufman, D. A. S., Sozda, C. N., Dotson, V. M., & Perlstein, W. M. (2016). An event-related potential investigation of the effects of age on alerting, orienting, and executive function. *Frontiers in Aging Neuroscience, 8*, 99. https://doi.org/10.3389/fnagi.2016.00099

Kirchner, W. K. (1958). Age differences in short-term retention of rapidly changing information. *Journal of Experimental Psychology, 55*(4), 352–358.

Logan, G. D., & Cowan, W. B. (1984). On the ability to inhibit thought and action: A theory of an act of control. *Psychological Review, 91*(3), 295–327.

Rueda, M. R., Fan, J., McCandliss, B. D., Halparin, J. D., Gruber, D. B., Lercari, L. P., & Posner, M. I. (2004). Development of attentional networks in childhood. *Neuropsychologia, 42*(8), 1029–1040.

Sedaghat, D., Lee, E., Garcia, J., & Brazendale, K. (2025). Screen time and objectively measured sleep of U.S. college students: A brief report. *International Journal of Exercise Science, 18*(4), 747–756. https://doi.org/10.70252/XDUT8341

Shallice, T. (1982). Specific impairments of planning. Philosophical Transactions of the Royal Society of London. Series B, Biological Sciences, 298(1089), 199–209. https://doi.org/10.1098/rstb.1982.0082

Stanislaw, H., & Todorov, N. (1999). Calculation of signal detection theory measures. *Behavior Research Methods, Instruments, & Computers, 31*(1), 137–149. https://doi.org/10.3758/BF03207704

Verbruggen, F., Aron, A. R., Band, G. P. H., Beste, C., Bissett, P. G., Brockett, A. T., et al. (2019). A consensus guide to capturing the ability to inhibit actions and impulsive behaviors in the stop-signal task. *eLife, 8*, e46323.

The game's own content (the insights shown while playing) has further sources; they are listed in `README.md` and in the game under "References".